# 01 · Business & Data Understanding  (CRISP-DM phases 1–2)

**Project:** Province-level, one-month-ahead food price forecasting for Zambia (WFP Food Prices dataset).

## Phase 1 – Business understanding
| Item | Decision |
|---|---|
| Business goal | Give planners/NGOs a quick next-month price outlook for each commodity in each province |
| Data-mining goal | Time-series regression: predict next month's price for every *province × commodity* series, with a lower/upper bound |
| Forecast target | price in month *t+1*, where *t* is the **last observation** of that series (e.g. last Beans/Central record = Jun-2025 → forecast Jul-2025) |
| Success criterion | Error under **10 %**. We read this as **MAPE < 10 %** (mean absolute *percentage* error) and also report MAE in ZMW |
| Deployment | FastAPI service: `/health`, `/commodities`, `/predict` |

Phase 2 below explores the data to find out what is (and is not) feasible.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
import prep
FIG = os.path.join(ROOT, "reports", "figures"); os.makedirs(FIG, exist_ok=True)
RAW = os.path.join(ROOT, "data", "wfp_food_prices_zmb.csv")
PROC = os.path.join(ROOT, "data", "processed"); os.makedirs(PROC, exist_ok=True)
MODELS = os.path.join(ROOT, "models"); os.makedirs(MODELS, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.autolayout": True})
def save(name):
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show()


## 2.1 Load and describe the raw data

In [ ]:
raw = pd.read_csv(RAW, parse_dates=["date"])
print(raw.shape)
display(raw.head())
display(raw.describe(include="all").T[["count","unique","top","min","max"]])
print("missing values per column:\n", raw.isna().sum()[raw.isna().sum()>0])

**Observations** – no missing cells, one currency (ZMW), only *Retail* prices, 10 provinces (`admin1`; note WFP spells Muchinga as *Muchiga*). The data are market-level, so we must **aggregate to province level**.

## 2.2 Coverage over time – the dataset is very uneven

In [ ]:
raw["year"] = raw.date.dt.year
fig, ax = plt.subplots(figsize=(9,3.5))
raw.groupby("year").size().plot.bar(ax=ax, color="#2a6f97")
ax.set_title("Number of price records per year"); ax.set_ylabel("records")
save("01_records_per_year.png")

In [ ]:
pivot = raw.pivot_table(index="commodity", columns="year", values="price", aggfunc="size", fill_value=0)
fig, ax = plt.subplots(figsize=(11,8))
im = ax.imshow(np.log1p(pivot.values), aspect="auto", cmap="Blues")
ax.set_yticks(range(len(pivot))); ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=90, fontsize=7)
ax.grid(False); ax.set_title("Records per commodity and year (log scale)"); plt.colorbar(im, label="log(1+records)")
save("01_commodity_year_heatmap.png")

**Observations** – two survey eras: 2012-2022 (staples, *Beans (dry)*, *Rice (local)*, …) and Nov-2023 onward (a broader basket: *Beans*, *Rice*, eggs, sugar, …). Several commodities exist for only a short window (Cassava meal, Millet, Sorghum end 2016).

## 2.3 Naming and unit inconsistencies

In [ ]:
units = raw.groupby(["commodity","unit"]).agg(first=("date","min"), last=("date","max"), n=("price","size"), median_price=("price","median")).reset_index()
display(units)

**Data-quality findings that drive Phase 3 decisions**
1. *Beans (dry)* (2012–2022) and *Beans* (2023–2025) are the same product → **merge**; likewise *Rice (local)* → *Rice*.
2. The same commodity is quoted in different pack sizes (maize meal 25 KG vs KG, oil 750 ML / 2.5 L / 5 L, …). Prices are **converted to a standard unit** (per KG / per L) so they can be modelled together.
3. Clear typos exist (milk at **ZMW 12,000**) → aggregate with the **median** instead of the mean.
4. Some series have a gap of >1 year between the two survey eras → we do **not** impute across such gaps; the series is split.

In [ ]:
display(raw.sort_values("price", ascending=False).head(6)[["date","admin1","market","commodity","unit","price"]])
print(raw.priceflag.value_counts())

## 2.4 Price dynamics and seasonality

In [ ]:
h = prep.harmonise(raw)
key = ["Maize (white)","Beans","Maize meal (white, roller)","Rice"]
fig, axes = plt.subplots(2,2, figsize=(11,6))
for ax, c in zip(axes.ravel(), key):
    s = h[h.commodity_std==c].groupby(["unit_std","month"]).price_std.median().reset_index()
    for u, g in s.groupby("unit_std"):
        ax.plot(g.month, g.price_std, lw=1.2, label=u)
    ax.set_title(f"{c} – national median price (ZMW per std unit)"); ax.legend(fontsize=7)
save("01_price_trends_key_commodities.png")

In [ ]:
# seasonality of white maize: month-of-year deviation from yearly mean (log)
m = h[(h.commodity_std=="Maize (white)")&(h.unit_std=="KG")].groupby("month").price_std.median()
lm = np.log(m); dev = lm - lm.rolling(12, center=True, min_periods=6).mean()
seas = dev.groupby(dev.index.month).mean()
fig, ax = plt.subplots(figsize=(7,3.2))
ax.bar(seas.index, seas.values*100, color="#8fb339"); ax.set_xticks(range(1,13))
ax.set_title("White maize: average seasonal deviation from trend (%)"); ax.set_xlabel("month")
save("01_maize_seasonality.png")

**Observations** – strong inflation trend (white maize ~ZMW 0.7/kg in 2003 → 3.6 in 2022) and a visible seasonal cycle (lean-season highs before the harvest). Therefore the model works on **log price changes** (scale-free) and includes calendar/seasonal features.

## 2.5 Province panel feasibility

In [ ]:
agg = prep.aggregate_province(h)
s = agg.groupby(["province","commodity","unit"]).agg(n=("price","size"), first=("month","min"), last=("month","max")).reset_index()
s["span"] = (s["last"].dt.year-s["first"].dt.year)*12 + s["last"].dt.month - s["first"].dt.month + 1
s["coverage"] = s.n/s.span
print("province-commodity-unit series:", len(s))
fig, ax = plt.subplots(1,2, figsize=(10,3.3))
s.n.plot.hist(bins=30, ax=ax[0], color="#2a6f97"); ax[0].set_title("Observed months per series")
s.coverage.plot.hist(bins=20, ax=ax[1], color="#e07a5f"); ax[1].set_title("Share of months observed inside series span")
save("01_series_length_and_coverage.png")
print((s.n>=12).sum(), "series have >= 12 monthly observations")

**Conclusion of Phase 2** – Short series (many have just ~20 months) mean a *single model per series* (ARIMA/ETS) would be unreliable. We will train **one pooled regression model** across all provinces and commodities on lag/seasonal features of the log-price, which also lets sparse series borrow strength from the others. Proceed to notebook 02.